## Session 2 · Topic 7 — Combining Tables: Multiple Sheets and Merge (Python)

**Dataset:** `contoso_aus.xlsx` — the **same workbook you used in the Excel case
study** (Contoso Australia). It has five sheets: `Sales`, `Product`, `Customer`,
`Store` and `Date`.

In Excel you worked across these sheets by hand: on **Day 1** you filled in the
calculated columns (revenue, cost, margin), and on **Day 2** you used **XLOOKUP**
to pull each product's name and category into the `Sales` sheet. This notebook
does the same two jobs in pandas:

1. **Read more than one sheet** into separate DataFrames.
2. **Rebuild a few Day 1 calculated columns** as vectorised operations.
3. **Merge** `Sales` with `Product` — pandas' version of a **SQL join**, and the
   direct equivalent of Day 2's XLOOKUP.

Replace every `# TODO` with your own code and run the cell.

### 1. Read more than one table

`pd.read_excel()` reads **one sheet at a time** — pass `sheet_name=` to choose
which. Read the `Sales` sheet into `sales` and the `Product` sheet into
`product`, then print the shape of each.

> The `Sales` sheet has empty columns (`ProductName`, `LineRevenue`, `Margin`, …)
> — those are the ones you filled *by hand* in Excel. Here you'll rebuild them,
> so ignore them for now.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("data")
DATA_FILE = DATA_FOLDER / "contoso_aus.xlsx"

# TODO: read the "Sales" sheet into df_sales
# TODO: read the "Product" sheet into df_product
# TODO: print the shape of each

### 2. Keep the base columns of Sales

Reduce `sales` to the columns that actually hold data — the ones every order line
comes with. Overwrite `sales` with just these:

`OrderKey`, `LineNumber`, `OrderDate`, `ProductKey`, `CustomerKey`, `StoreKey`,
`Quantity`, `UnitPrice`, `NetPrice`, `UnitCost`

Print `sales.head()`.

In [ ]:
base_cols = ["OrderKey", "LineNumber", "OrderDate", "ProductKey",
             "CustomerKey", "StoreKey", "Quantity", "UnitPrice",
             "NetPrice", "UnitCost"]
# TODO: keep only base_cols in df_sales
# TODO: show df_sales.head()

### 3. Rebuild the Day 1 calculated columns (vectorised)

In Excel you wrote one formula and filled it down thousands of rows. In pandas
you write the expression **once** and it applies to the **whole column at once** —
this is a *vectorised* operation, the pandas equivalent of filling a column down.

Create these five columns (same formulas as the Excel case study):

- `GrossExtension = Quantity * UnitPrice`
- `LineRevenue    = Quantity * NetPrice`
- `LineCost       = Quantity * UnitCost`
- `Margin         = LineRevenue - LineCost`
- `MarginPercent  = Margin / LineRevenue`

Then show `sales[["Quantity", "NetPrice", "LineRevenue", "Margin", "MarginPercent"]].head()`.

In [ ]:
# TODO: GrossExtension = Quantity * UnitPrice
# TODO: LineRevenue    = Quantity * NetPrice
# TODO: LineCost       = Quantity * UnitCost
# TODO: Margin         = LineRevenue - LineCost
# TODO: MarginPercent  = Margin / LineRevenue
# TODO: show a few columns to check

### 4. Connecting tables — a merge is a SQL join

`sales` tells you *what* was sold (a `ProductKey`) but not the product's **name**,
**category** or **brand** — those live in the `product` table. Connecting them is
exactly a **join**, which you've seen in SQL:

```sql
SELECT s.*, p.ProductName, p.CategoryName, p.Brand
FROM Sales s
JOIN Product p ON s.ProductKey = p.ProductKey
```

In pandas the same operation is `merge`:

```
sales.merge(product_lookup, on="ProductKey", how="inner")
```

- **`on="ProductKey"`** — the join key: the column both tables share (the SQL
  `ON` clause).
- **`how=`** — the join type: `inner` (only matching keys, like SQL `INNER JOIN`),
  `left` (keep every row of the left table, like `LEFT JOIN`), `right`, or
  `outer` (like `FULL OUTER JOIN`).
- `product` has **one** row per `ProductKey`; `sales` has **many** rows sharing
  each key — a **many-to-one** join. Every sales row picks up its product's
  details, which is precisely what XLOOKUP did in Excel Day 2.

```
   Sales (many)                     Product (one per key)
   ProductKey ─────────────────────▶ ProductKey, ProductName, CategoryName, Brand
```

First, take just the columns you need from `product` into `product_lookup`:
`ProductKey`, `ProductName`, `CategoryName`, `Brand`. Pulling only what you need
keeps the result tidy (a full merge would drag in all 14 product columns).

In [ ]:
# TODO: select ProductKey, ProductName, CategoryName, Brand from df_product
#       into df_product_lookup, then show its head()

### 5. Do the merge

Merge `sales` with `product_lookup` on `ProductKey` using an **inner** join, and
assign the result back to `sales`.

Print the shape before and after. Because every `ProductKey` in `sales` exists in
`product`, and `product` has one row per key, the row count should **stay the
same** (7,739) — you've only added columns, not rows. Then show
`sales[["ProductKey", "ProductName", "CategoryName", "Brand", "LineRevenue"]].head()`.

In [ ]:
# TODO: print df_sales.shape (before)
# TODO: merge sales with product_lookup on "ProductKey", how="inner", back into sales
# TODO: print sales.shape (after) — did the row count change?
# TODO: show ProductKey, ProductName, CategoryName, Brand, LineRevenue

### 6. The payoff — a question you couldn't answer before

Before the merge you only had a `ProductKey`; now you have the category. So you
can finally ask a business question: **which category earns the most revenue?**

Group by `CategoryName`, sum `LineRevenue`, and sort highest to lowest.

> This is a preview of Session 3's `groupby()` — here it just shows *why* the
> merge was worth doing.

In [ ]:
# TODO: groupby CategoryName, sum LineRevenue, sort descending

### 7. Wrap-up

In your own words (2–3 sentences): what does a `merge` do, and which SQL operation
is it the same as? Why did the row count stay at 7,739 after the merge instead of
growing? And how is this merge the same idea as the XLOOKUP you wrote in Excel
Day 2?